# Dimensionality reduction using PCA

In [ ]:
from glob import glob
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(data.columns[:12], axis=1)

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop("geometry")
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    scaler = StandardScaler()
    data_relative[cols_to_normalize] = scaler.fit_transform(
        data_relative[cols_to_normalize]
    )
    data_relative[cols_no_division] = scaler.fit_transform(
        data_relative[cols_no_division]
    )

    return data_relative

In [ ]:
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"

In [ ]:
data_relative = process_file(path, path_total)

## Apply dimensionality reduction

In [ ]:
pca = PCA()
pca.fit(data_relative.drop(columns=["Obyvatelstvo celkem", "geometry"]))

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.axhline(0.8, color="r", linestyle="--")
plt.grid()
plt.show()

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_variance >= 0.8)

In [ ]:
n_components

In [ ]:
pca = PCA(n_components=n_components)
pca.fit(data_relative.drop(columns=["Obyvatelstvo celkem", "geometry"]))

In [ ]:
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loadings_df = pd.DataFrame(
    loadings.T,
    index=[f"PC{i + 1}" for i in range(pca.explained_variance_.shape[0])],
    columns=(data_relative.drop(columns=["Obyvatelstvo celkem", "geometry"])).columns,
)
loadings_df.T.style.background_gradient(cmap="RdBu", vmin=-1, vmax=1)

## Interpret the loadings
Create a df with the highest positive and negative loadings for each component

In [ ]:
df = loadings_df.T
summary = {}

for col in df.columns:
    top5 = df[col].nlargest(5)
    bottom5 = df[col].nsmallest(5)

    # Combine top and bottom into one mini DataFrame
    summary[col] = pd.DataFrame(
        {
            "Positive": top5.values,
            "Positive variables": top5.index,
            "Negative": bottom5.values,
            "Negative variables": bottom5.index,
        }
    )

loadings_df = pd.concat(summary, axis=0)

loadings_df

In [ ]:
loadings_df.T.to_csv("/data/uscuni-restricted/05_pcs/loadings_pca.txt")

In [ ]:
transformed = pca.transform(
    data_relative.drop(columns=["Obyvatelstvo celkem", "geometry"])
)

df_pca = pd.DataFrame(
    transformed,
    index=data_relative.drop(columns=["Obyvatelstvo celkem", "geometry"]).index,
).set_geometry(data_relative.geometry)

df_pca.columns = df_pca.columns.astype(str)
df_pca.to_parquet("/data/uscuni-restricted/05_pcs/pca_autumn.parquet")